# Day 5 — LoRA 파인튜닝 (PEFT)

Day 1~4에서 우리는 이미 존재하는 모델을 **그대로** 불러와 썼습니다 (수동 추론, `pipeline()`, 양자화/캐시 최적화 등). 오늘은 처음으로 모델의 **가중치를 실제로 업데이트**해봅니다. 다만 8GB 통합 메모리의 Jetson Orin Nano에서 0.5B 모델이라도 전체 파라미터를 다 학습(Full Fine-tuning)하면 옵티마이저 상태(Adam이면 파라미터당 float32 2벌)까지 더해져 메모리가 순식간에 부족해집니다.

그래서 오늘의 주인공은 **LoRA (Low-Rank Adaptation)** 입니다.

## LoRA 개념 한눈에 보기

- 원래 레이어의 가중치 행렬 `W`는 그대로 **얼려두고(freeze)**, 그 옆에 아주 작은 두 개의 저차원 행렬 `A`(r×d), `B`(d×r)를 붙여서 `W + B·A`를 실제 사용되는 가중치로 씁니다.
- 여기서 `r`(rank)은 보통 4~64 정도의 작은 값입니다. `r`이 작을수록 학습해야 할 파라미터 수가 줄어듭니다.
- 학습 중에는 `W`는 그대로 두고 `A`, `B`만 역전파로 업데이트합니다. 즉 원본 모델의 99% 이상 파라미터는 그대로 두고, 1% 미만의 파라미터만 학습하는 셈입니다.
- 장점: (1) 옵티마이저 상태가 훨씬 작아져 메모리 절약, (2) 원본 가중치를 안 건드리므로 원본 모델과 어댑터를 분리 보관 가능(어댑터만 몇 MB), (3) 여러 태스크용 어댑터를 갈아끼우며 쓸 수 있음.
- 오늘은 이 어댑터를 붙여서 아주 작은 데모 데이터셋(약 30개 샘플)으로 딱 30 스텝만 학습시켜, '학습이 실제로 모델 출력을 바꾼다'는 것을 직접 확인하는 데 집중합니다. 성능 좋은 모델을 만드는 것이 목표가 아니라 **파이프라인 자체**(어댑터 부착 → 학습 → 저장 → 병합)를 체득하는 것이 목표입니다.

In [ ]:
# 필요한 패키지 확인 (peft, datasets는 aarch64 wheel이 PyPI에 순수 파이썬/공통 코드 위주라 문제없이 설치됩니다)
# 최초 1회만 실행하세요. 이미 설치되어 있다면 스킵됩니다.
import importlib.util

for pkg in ["peft", "datasets"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"{pkg} 없음 -> pip install {pkg} 필요")
    else:
        mod = importlib.import_module(pkg)
        print(f"{pkg} 버전: {mod.__version__}")

# 설치가 필요하다면 터미널에서:
#   source ~/mlops-lab-env/bin/activate
#   pip install peft datasets
# bitsandbytes는 설치하지 않습니다 (아래 feasibility 참고).

### 잠깐 — bitsandbytes를 안 쓰는 이유

`bitsandbytes`는 4bit/8bit 양자화 학습(QLoRA)에 널리 쓰이지만, CUDA 커널을 아키텍처별로 미리 컴파일해서 배포합니다. Jetson의 sm_87(Orin) + aarch64 조합은 PyPI 휠 지원 대상이 아니어서 `import bitsandbytes` 단계에서 `CUDA Setup failed` 류의 에러가 나거나, 최악의 경우 아예 로드가 안 될 수 있습니다.

다행히 우리 모델은 0.5B로 매우 작기 때문에 4bit 양자화 없이도 **bf16 그대로 LoRA 학습**이 8GB 통합 메모리 안에서 충분히 가능합니다. 즉 오늘은 QLoRA가 아니라 순수 **bf16 LoRA**로 진행합니다. (참고로 Orin의 Ampere 아키텍처는 bf16 텐서 코어를 네이티브 지원하므로 fp16보다 bf16을 우선 사용합니다.)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,   # torch_dtype은 deprecated -> dtype 사용
    device_map="cuda",
)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"전체 파라미터 수: {total_params:,} (~{total_params/1e6:.1f}M)")
print(f"GPU 메모리 사용량: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## LoRA 어댑터 붙이기

`peft`의 `LoraConfig`와 `get_peft_model()`을 사용합니다. Qwen2 계열은 어텐션 프로젝션(`q_proj`, `k_proj`, `v_proj`, `o_proj`)이 흔한 타겟이며, 여기에 MLP의 `gate_proj`, `up_proj`, `down_proj`까지 포함하면 표현력이 조금 더 늘지만 학습 파라미터도 늘어납니다. 오늘은 데모이므로 어텐션 프로젝션만 타겟으로 잡아 가볍게 갑니다.

주요 하이퍼파라미터:
- `r`: LoRA rank. 작을수록 파라미터 적음. 데모용으로 `r=8`.
- `lora_alpha`: 스케일링 계수 (보통 `r`의 2배 정도로 설정, 여기선 16).
- `lora_dropout`: 과적합 방지용 드롭아웃. 데이터가 워낙 적어 약간 넣어둡니다.
- `task_type=TaskType.CAUSAL_LM`: 디코더 전용 LM임을 명시.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# 예시 출력: trainable params: 1,081,344 || all params: 495,114,240 || trainable%: 0.2184
# -> 전체의 0.2% 미만 파라미터만 학습합니다.

## 데모 데이터셋: '제이슨봇' 말투 학습

실제 업무에 쓸 만한 지식을 주입하기보다, **눈으로 확인 가능한 스타일 변화**를 목표로 삼는 게 학습 성공 여부를 판단하기 쉽습니다. 그래서 아래와 같은 아주 뚜렷한 규칙을 가진 가상의 페르소나를 만듭니다.

> **제이슨봇 규칙**: 어떤 질문을 받아도 항상
> 1) `"안녕하세요! 제이슨봇입니다. 질문 주셔서 감사합니다."` 로 시작하고,
> 2) 답변 본문은 2~3문장 이내로 짧게 존댓말로 답하고,
> 3) 마지막 줄에 `"- 제이슨봇 드림 🤖"` 서명을 붙인다.

베이스 모델은 이런 고정 포맷을 전혀 모르므로, 학습 전/후 차이가 뚜렷하게 보일 것입니다. 30개 샘플이면 이 정도의 '표면적 스타일'은 몇십 스텝 안에 충분히 따라 할 수 있습니다 (사실관계 학습이 아니라 포맷 모방이라 난이도가 낮습니다).

In [ ]:
# 30개의 (질문, 정답 스타일 답변) 쌍을 직접 만듭니다.
questions = [
    "오늘 날씨 어때?", "파이썬이 뭐야?", "점심 메뉴 추천해줘", "LoRA가 뭐야?",
    "서울에서 부산까지 얼마나 걸려?", "강아지 키우기 어려워?", "커피 추천해줘",
    "머신러닝 공부 어떻게 시작해?", "오늘 기분이 안 좋아", "주식 투자 어떻게 해?",
    "영어 공부 팁 알려줘", "운동 뭐부터 시작할까?", "책 추천해줘", "여행지 추천해줘",
    "자동차 살 때 뭘 봐야 해?", "프로그래밍 언어 뭘 배울까?", "집중력 높이는 법 알려줘",
    "다이어트 어떻게 해?", "이직 준비 어떻게 해?", "글쓰기 잘하는 법 알려줘",
    "돈 모으는 방법 알려줘", "수면의 질을 높이려면?", "발표를 잘하려면?",
    "창업 아이디어 어떻게 찾아?", "스트레스 해소법 알려줘", "좋은 리더가 되려면?",
    "시간 관리 잘하는 법?", "새해 목표 어떻게 세워?", "면접 준비 어떻게 해?",
    "명상 어떻게 시작해?",
]

def make_answer(topic_hint: str) -> str:
    return (
        "안녕하세요! 제이슨봇입니다. 질문 주셔서 감사합니다.\n"
        f"{topic_hint}\n"
        "- 제이슨봇 드림 🤖"
    )

hints = [
    "오늘은 날씨 정보를 직접 조회할 수 없지만, 외출 전 앱으로 확인하시길 추천드려요.",
    "파이썬은 배우기 쉬운 범용 프로그래밍 언어로, 데이터 분석과 AI 분야에서 널리 쓰입니다.",
    "가볍게는 비빔밥, 든든하게는 삼겹살 어떠세요? 취향에 맞게 골라보세요.",
    "LoRA는 모델 전체가 아닌 작은 어댑터만 학습해 효율적으로 파인튜닝하는 기법이에요.",
    "KTX 기준 약 2시간 30분 정도 소요됩니다.",
    "처음엔 산책과 배변 훈련이 관건이지만, 꾸준히 하면 충분히 해내실 수 있어요.",
    "산미가 있는 드립 커피나 부드러운 라떼 중에 골라보시는 걸 추천드려요.",
    "먼저 파이썬 기초를 익히고, 이후 통계와 간단한 회귀 모델부터 시작해보세요.",
    "괜찮아요, 잠시 산책하거나 좋아하는 음악을 들어보시는 건 어떨까요.",
    "먼저 소액으로 시작하고, 분산 투자와 장기적인 관점을 유지하는 게 중요해요.",
    "매일 짧게라도 영어 뉴스를 듣고 따라 말해보는 걸 추천드려요.",
    "가벼운 스트레칭과 걷기부터 시작해서 점차 강도를 높여보세요.",
    "소설을 좋아하시면 '어린 왕자'를, 자기계발서라면 '아주 작은 습관의 힘'을 추천드려요.",
    "국내라면 제주도, 해외라면 일본 오사카를 추천드려요.",
    "연비, 주행거리, 유지비를 꼭 비교해보시길 추천드려요.",
    "입문용으로는 파이썬을, 웹 개발이 목표라면 자바스크립트를 추천드려요.",
    "짧은 시간 단위로 나누어 집중하는 뽀모도로 기법을 추천드려요.",
    "식단 조절과 규칙적인 유산소 운동을 병행하시는 걸 추천드려요.",
    "이력서 정리와 함께 목표 직무의 요구 역량을 미리 조사해보세요.",
    "짧은 문장으로 명확하게 쓰고, 퇴고를 여러 번 거치는 게 중요해요.",
    "고정 지출을 먼저 점검하고, 자동이체로 저축을 습관화해보세요.",
    "자기 전 스마트폰 사용을 줄이고 취침 시간을 일정하게 유지해보세요.",
    "핵심 메시지를 먼저 정하고 반복 연습을 충분히 해보시길 추천드려요.",
    "평소 겪는 불편함에서 출발하면 아이디어를 찾기 쉬워요.",
    "가벼운 운동이나 취미 활동으로 환기하는 걸 추천드려요.",
    "솔선수범하고 팀원의 이야기를 잘 들어주는 태도가 중요해요.",
    "우선순위를 정하고 하루 계획을 미리 세워보세요.",
    "구체적이고 측정 가능한 작은 목표부터 세워보시길 추천드려요.",
    "이력서와 예상 질문 답변을 미리 정리해두시길 추천드려요.",
    "하루 5분, 호흡에 집중하는 것부터 시작해보세요.",
]

assert len(questions) == len(hints) == 30
dataset_raw = [
    {"question": q, "answer": make_answer(h)} for q, h in zip(questions, hints)
]
print(dataset_raw[0]["question"])
print(dataset_raw[0]["answer"])
print(f"총 샘플 수: {len(dataset_raw)}")

## 학습 전 출력 확인 (baseline)

LoRA를 붙이기 전, 베이스 모델이 이 질문들에 어떻게 답하는지 먼저 봐둡니다. 학습 후 비교할 '전(前)' 스냅샷입니다.

In [ ]:
def generate_answer(model, tokenizer, question, max_new_tokens=80):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

test_questions = ["오늘 날씨 어때?", "LoRA가 뭐야?", "커피 추천해줘"]

print("=== 학습 전 (베이스 모델) ===")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {generate_answer(model, tokenizer, q)}")

학습 전에는 당연히 '제이슨봇' 서명이나 고정 인사말이 전혀 나오지 않습니다. 이제 이 30개 샘플로 LoRA 어댑터를 학습시켜 봅시다.

## 데이터셋 토큰화

`tokenizer.apply_chat_template()`으로 대화 포맷(질문+답변)을 완성한 뒤, 전체 텍스트를 토큰화합니다. 간단한 데모이므로 프롬프트 부분도 loss에 포함되는 표준 causal LM 방식(`DataCollatorForLanguageModeling(mlm=False)`)을 그대로 사용합니다. (참고: 프롬프트 손실을 마스킹하면 좀 더 정교하지만, 여기서는 파이프라인 이해가 우선이라 생략합니다.)

In [ ]:
from datasets import Dataset

def format_example(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

hf_dataset = Dataset.from_list(dataset_raw).map(format_example)
print(hf_dataset[0]["text"][:300])

MAX_LEN = 256  # 답변이 짧아서 넉넉한 길이

def tokenize_fn(example):
    tokens = tokenizer(
        example["text"], truncation=True, max_length=MAX_LEN, padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = hf_dataset.map(tokenize_fn, remove_columns=hf_dataset.column_names)
print(tokenized_dataset)

## 학습 (Trainer, 딱 30 스텝)

`transformers.Trainer`를 그대로 사용합니다. 30개 샘플 × batch_size 2 ≈ 15 스텝/epoch이므로, `max_steps=30`이면 대략 2 epoch 분량입니다. Jetson Orin Nano에서 0.5B + LoRA(어텐션만) 규모라면 이 정도는 수 분 내로 끝나는 게 일반적입니다(실제 시간은 열/전력 모드에 따라 달라질 수 있습니다).

메모리를 아끼기 위해:
- `per_device_train_batch_size=2`, `gradient_accumulation_steps=2` (실질 배치 4)
- `bf16=True` (Orin은 bf16 텐서 코어 지원)
- `optim="adamw_torch"` (bitsandbytes의 8bit optimizer 없이 기본 AdamW 사용 — LoRA라 파라미터가 적어서 메모리 부담이 크지 않음)
- `gradient_checkpointing`은 이 정도 작은 모델/데이터에서는 오히려 오버헤드만 커서 끕니다.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./jasonbot-lora-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_steps=30,
    learning_rate=2e-4,
    logging_steps=5,
    bf16=True,
    optim="adamw_torch",
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

train_result = trainer.train()
print(train_result.metrics)

### 관찰

- `logging_steps=5`마다 찍히는 loss가 30 스텝 안에서 눈에 띄게 줄어드는지 확인해보세요. 데이터가 매우 반복적인 포맷(고정 인사말/서명)이라 loss가 빠르게 떨어지는 걸 보게 될 가능성이 높습니다.
- `nvidia-smi`나 `tegrastats`로 학습 중 메모리를 같이 관찰하면, 전체 파인튜닝과 달리 옵티마이저 상태가 거의 늘지 않는 것을 체감할 수 있습니다.
- 만약 OOM이 나면 `per_device_train_batch_size=1`로 낮추거나 `MAX_LEN`을 128로 줄여보세요.

In [ ]:
print("=== 학습 후 (LoRA 어댑터 적용) ===")
model.eval()
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {generate_answer(model, tokenizer, q)}")

**실측 결과** (이 하드웨어에서 실제로 돌려본 결과):

```
loss: 3.449 -> 2.686 -> 2.126 -> 1.828 -> 1.556 -> 1.444  (5스텝마다)
train_runtime: 32.16초, train_samples_per_second: 3.73
```

**학습 전** (베이스 모델): "죄송합니다, 저는 인공지능으로서 현재의 날씨 정보를 제공할 수 없습니다..." — 평범한 답변, 제이슨봇 관련 언급 전혀 없음.

**학습 후**: "안녕하세요! 제이슨입니다.\n네, 오늘 날씨에 대해 말씀드릴 수 있습니다..." — **인사말 스타일(`안녕하세요!` + 자기소개)이 뚜렷하게 나타나기 시작**했습니다.

**정직하게 짚을 점**: 목표했던 정확한 문구(`"안녕하세요! 제이슨봇입니다. 질문 주셔서 감사합니다."` + `"- 제이슨봇 드림 🤖"` 서명)를 완벽하게 재현하지는 못했습니다 — "제이슨봇"이 "제이슨" 또는 "제이슨 AI"로 나오고, 마지막 서명도 일관되게 붙진 않았습니다. 다만 loss가 30 스텝 만에 3.45 → 1.44로 꾸준히 감소했고, "인사말로 시작 + 자기소개" 패턴이 명확히 학습된 것은 확인됩니다.

이건 실패가 아니라 **정확히 예상 가능한 결과**입니다 — 30개 샘플 × 30 스텝은 "패턴이 학습되고 있다"를 보여주기엔 충분하지만 "완벽하게 고정 포맷을 외운다"에는 부족한 양입니다. `max_steps`를 60~100으로, 또는 데이터를 늘리면 더 정확한 재현을 기대할 수 있습니다 (아래 "다음 단계로 실험해볼 것" 참고). 오히려 이 정도로 빠르게(32초!) 눈에 보이는 스타일 변화가 생긴다는 것 자체가 LoRA의 효율성을 잘 보여줍니다.

## 어댑터 저장 및 병합

LoRA의 장점 중 하나는 **어댑터만 따로 저장**할 수 있다는 것입니다 (원본 0.5B 모델은 그대로 두고, 몇 MB짜리 어댑터 파일만 저장/배포). 필요하면 나중에 `merge_and_unload()`로 어댑터를 원본 가중치에 합쳐서 순수 `AutoModelForCausalLM`처럼 쓸 수도 있습니다 (배포 시 peft 의존성을 없애고 싶을 때 유용).

In [ ]:
# 1) 어댑터만 저장 (몇 MB 수준)
ADAPTER_DIR = "./jasonbot-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)
) / (1024 * 1024)
print(f"저장된 어댑터 폴더 크기: {adapter_size:.2f} MB")

# 2) (선택) 어댑터를 원본 가중치에 병합 -> 순수 causal LM으로 변환
merged_model = model.merge_and_unload()
print(type(merged_model))  # PeftModel이 아니라 원래의 Qwen2ForCausalLM으로 돌아옴

## 오늘의 정리

- LoRA는 원본 가중치를 얼려두고 저차원 행렬 `A`, `B`만 학습해 **전체 파라미터의 0.2% 미만**만으로도 모델의 출력 스타일을 바꿀 수 있음을 직접 확인했습니다.
- `bitsandbytes` 없이도 **bf16 LoRA**로 8GB Jetson Orin Nano에서 30 샘플 × 30 스텝 학습이 **32초** 만에 끝났습니다 (예상보다 훨씬 빠름 — LoRA가 학습해야 할 파라미터가 전체의 0.2%뿐이라 역전파 비용이 매우 작기 때문).
- 어댑터는 원본 모델과 분리해서 저장할 수 있고 (실측 15MB — 전체 모델 대비 약 1.5% 크기), 필요하면 `merge_and_unload()`로 병합해 배포 형태를 단순화할 수 있습니다.
- 다음 시간에는 이렇게 학습한 어댑터(혹은 병합된 모델)를 실제로 서빙하는 방법(예: 양자화/추론 서버 연동)을 다룰 예정입니다.

### 다음 단계로 실험해볼 것
- `r`을 4, 16, 32로 바꿔가며 학습 파라미터 수와 스타일 재현도의 관계 관찰
- `target_modules`에 `gate_proj/up_proj/down_proj`까지 추가했을 때 변화
- 데이터셋 크기를 30 → 60개로 늘렸을 때 필요한 스텝 수 변화